# Nigerian Forensic Stylistics Pipeline

**Paper:** *The Computable Voice: Can a Machine Speak Like Awaisu?*
**Author:** Rabiu Raji ([ORCID 0009-0007-8968-8620](https://orcid.org/0009-0007-8968-8620))

**Goal:** Test whether LLM-generated imitations of four contemporary Nigerian authors (Awaisu, Egya, Liam, Shittu) can be distinguished from the originals using Burrows' Delta, Biber features, and classifier feature importance.

**Auto-loaded corpus (no upload needed):** Corpus fetched directly from GitHub raw URLs. 27 passages across 4 authors, ~13K words.

**Pipeline (6 cells):**
1. **Install** dependencies (pybiber, nltk, scikit-learn, etc.)
2. **Load human corpus** from GitHub (auto-fetch)
3. **Generate LLM passages** (needs API key — Groq recommended, free)
4. **Feature extraction** (Biber's 67 features + lexical + syntactic + function-word)
5. **Stylometric analysis** (Burrows' Delta, hierarchical clustering, MDS)
6. **Classification** (Per-author binary classifier + SHAP feature importance)

**Hardware:** Free Colab CPU is fine. ~15-20 min runtime after corpus + API key are ready.

**Step 1:** Just run this cell to install everything. No setup needed.

## CELL 1/6: Install dependencies (1-2 minutes)

In [ ]:
import subprocess
import sys

# Core dependencies
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pybiber',           # Biber's 67-feature extraction
    'nltk',              # Tokenization
    'spacy',             # Dependency parsing
    'scikit-learn',      # Classification, clustering
    'shap',              # Feature importance
    'matplotlib',        # Visualizations
    'seaborn',           # Pretty plots
    'pandas',            # Data wrangling
    'numpy',
    'scipy',
], check=True)

# Spacy model for dependency parsing
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], capture_output=True)

import pybiber
import nltk
import spacy
import sklearn
import shap
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.manifold import MDS
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

print(f'pybiber: {pybiber.__version__ if hasattr(pybiber, "__version__") else "installed"}')
print(f'spacy: {spacy.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'shap: {shap.__version__}')
print(f'pandas: {pd.__version__}')


## CELL 2/6: Load human corpus

**Before running this cell:** upload `human_corpus.csv` with columns: `id, title, author, year, genre, text`

Each row = one 500-word excerpt from one novel. 60 rows = 60 excerpts.

In [ ]:
# Fetch corpus metadata directly from GitHub raw URL
import pandas as pd

GITHUB_RAW = "https://raw.githubusercontent.com/Rawbeew/nigerian-forensic-stylistics/main/corpus/metadata.csv"

print('Fetching metadata.csv from GitHub...')
human_df = pd.read_csv(GITHUB_RAW)
print(f'Loaded {len(human_df)} human passages')
print(f'Columns: {list(human_df.columns)}')

# Word count for each passage
human_df['wc'] = human_df['text'].astype(str).str.split().str.len()
print(f'\nWord count stats:')
print(human_df['wc'].describe())

# Filter to valid lengths and identify short ones
short = human_df[human_df['wc'] < 200].copy()
if len(short) > 0:
    print(f'\nNote: {len(short)} very short excerpts (<200 words):')
    for _, row in short.iterrows():
        print(f"  - {row['id']}: {row['wc']} words")
    print('  These will be combined/padded for analysis.')

# Per author breakdown
print(f'\nPer author:')
print(human_df.groupby('author').size())

# Standard: keep 200-1000 word range
human_df = human_df[human_df['wc'] >= 100].reset_index(drop=True)
print(f'\nAfter filtering: {len(human_df)} passages kept')

# Save locally for the next cells
human_df.to_csv('human_corpus.csv', index=False)
print('\nSaved as human_corpus.csv (used in next cells)')


## CELL 3/6: Generate LLM passages

Four models, three prompt conditions, 20 paragraphs each = 240 synthetic passages.

**API options (use any ONE that works):**
- OpenAI (GPT-4o, GPT-3.5)
- Anthropic (Claude 3.5 Sonnet)
- Google AI Studio (Gemini 1.5 Pro)
- Groq / OpenRouter (Llama 3-70B) - RECOMMENDED FREE
- All have free tiers sufficient for 240 passages

**Set your API keys below or use Colab Secrets.**

In [ ]:
import os
import time
import requests
import pandas as pd
import random

# 4-author configuration
AUTHORS = {
    'awaisu': {
        'genre': 'novel',
        'themes': "women's rights in Northern Nigeria, sickle cell disease, conservative Muslim family dynamics",
        'style_note': 'formal and colloquial mix, vivid imagery, complex sentences for introspection',
        'reference_works': 'The Thing About Compromise, Burning Bright, Ms. Joanas Rules',
        'id': 'awaisu',
    },
    'egya': {
        'genre': 'novel+poetry+criticism',
        'themes': 'environmental degradation, Niger Delta, political corruption, national identity',
        'style_note': 'dense figurative language, nature imagery, protest undertones',
        'reference_works': 'Sterile Sky, Makwala, What the Sea Told Me, Nation Power and Dissidence',
        'id': 'egya',
    },
    'liam': {
        'genre': 'poetry',
        'themes': 'barracks life, Tiv culture, religion and ethnic polarization, emerging northern voices',
        'style_note': 'compact lines, often direct, journalistic clarity',
        'reference_works': 'Indefinite Cravings, Saint Shaade and Other Poems',
        'id': 'liam',
    },
    'shittu': {
        'genre': 'poetry+drama',
        'themes': 'japa, migration, national disillusionment, historical memory',
        'style_note': 'time-spanning structure, metaphorical density, ironic tone',
        'reference_works': 'Niger Blues and other Poems, The Crash, Japa Elegy for Nigerians',
        'id': 'shittu',
    },
}

N_PER_AUTHOR_PROMPT = 10  # 10 passages per (author × model × prompt)

# Prompt templates (3 conditions per author)
def build_prompts(author_key, author_info):
    name = author_key.capitalize()
    return {
        'P1_generic': (
            f"Write 500 words of contemporary Nigerian {author_info['genre']} "
            f"in the style of {name}, a contemporary Nigerian writer. "
            f"Capture their typical sentence rhythm, vocabulary, and thematic preoccupations. "
            f"Reference works: {author_info['reference_works']}."
        ),
        'P2_genre_specific': (
            f"Write 500 words of {author_info['genre']} in the voice of {name}. "
            f"Style notes: {author_info['style_note']}. "
            f"Common themes: {author_info['themes']}."
        ),
        'P3_themed': (
            f"Write 500 words of contemporary Nigerian literary prose "
            f"about {author_info['themes']}, "
            f"in a style that resembles the rhythm and imagery of {name}. "
            f"Focus on the texture of ordinary Nigerian life and the tensions "
            f"between tradition and modernity."
        ),
    }

# Provider functions (same as before)
GROQ_API_KEY = ''
OPENROUTER_API_KEY = ''

def generate_groq(prompt, model='llama-3.1-70b-versatile', temperature=0.7):
    if not GROQ_API_KEY: return None
    try:
        r = requests.post(
            'https://api.groq.com/openai/v1/chat/completions',
            headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'User-Agent': 'Mozilla/5.0'},
            json={'model': model, 'messages': [{'role': 'user', 'content': prompt}],
                  'temperature': temperature, 'max_tokens': 750},
            timeout=60)
        if r.status_code == 200:
            return r.json()['choices'][0]['message']['content']
    except Exception as e: print(f'Error: {e}')
    return None

def generate_openrouter(prompt, model='meta-llama/llama-3-70b-instruct', temperature=0.7):
    if not OPENROUTER_API_KEY: return None
    try:
        r = requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}'},
            json={'model': model, 'messages': [{'role': 'user', 'content': prompt}],
                  'temperature': temperature, 'max_tokens': 750},
            timeout=60)
        if r.status_code == 200:
            return r.json()['choices'][0]['message']['content']
    except Exception as e: print(f'Error: {e}')
    return None

def generate(prompt, model='default'):
    for fn, m in [
        (lambda p: generate_groq(p), 'llama-3.1-70b'),
        (lambda p: generate_openrouter(p), 'llama-3-70b'),
    ]:
        result = fn(prompt)
        if result: return result, m
    return None, None

# Generate LLM corpus — 4 authors × 3 prompts × N passages
synthetic_records = []

for author_key, author_info in AUTHORS.items():
    print(f'\n=== Generating for {author_key.upper()} ===')
    prompts = build_prompts(author_key, author_info)
    for prompt_id, prompt_template in prompts.items():
        print(f'\n  {prompt_id} ({N_PER_AUTHOR_PROMPT} passages)...')
        for i in range(N_PER_AUTHOR_PROMPT):
            text, model_used = generate(prompt_template)
            if text:
                synthetic_records.append({
                    'id': f'synth_{author_key}_{prompt_id}_{i:02d}',
                    'author': author_key,
                    'prompt_id': prompt_id,
                    'model': model_used,
                    'text': text,
                })
                print(f'    [{i+1}/{N_PER_AUTHOR_PROMPT}] OK ({len(text.split())} words)')
            else:
                print(f'    [{i+1}/{N_PER_AUTHOR_PROMPT}] FAILED')
            time.sleep(0.5)

synth_df = pd.DataFrame(synthetic_records)
synth_df.to_csv('synthetic_corpus.csv', index=False)
print(f'\nSaved {len(synth_df)} synthetic passages to synthetic_corpus.csv')
print(f'Per author: {synth_df["author"].value_counts().to_dict()}')
print(f'Per model: {synth_df["model"].value_counts().to_dict()}')


## CELL 4/6: Feature extraction (Biber + lexical + syntactic)

In [ ]:
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize

def extract_features(text):
    'Extract lexical, function-word, syntactic, and Biber-like features.'
    features = {}

    tokens = word_tokenize(text.lower())
    sents = sent_tokenize(text)

    # Lexical
    features['wc'] = len(tokens)
    features['unique_words'] = len(set(tokens))
    features['ttr'] = len(set(tokens)) / len(tokens) if tokens else 0
    features['avg_word_len'] = np.mean([len(t) for t in tokens if t.isalpha()]) if tokens else 0

    # Sentence stats
    sent_lens = [len(word_tokenize(s)) for s in sents]
    features['n_sents'] = len(sents)
    features['mean_sent_len'] = np.mean(sent_lens) if sent_lens else 0
    features['sent_len_std'] = np.std(sent_lens) if sent_lens else 0
    features['max_sent_len'] = max(sent_lens) if sent_lens else 0

    # Function words (top 100 most common English)
    function_words = ['the','be','to','of','and','a','in','that','have','i','it','for','not','on','with','he','as','you','do','at','this','but','his','by','from','they','we','say','her','she','or','an','will','my','one','all','would','there','their','what','so','up','out','if','about','who','get','which','go','me','when','make','can','like','time','no','just','him','know','take','people','into','year','your','good','some','could','them','see','other','than','then','now','look','only','come','its','over','think','also','back','after','use','two','how','our','work','first','well','way','even','new','want','because','any','these','give','day','most']
    fw_counts = Counter(t for t in tokens if t in function_words)
    features['fw_total'] = sum(fw_counts.values())
    features['fw_diversity'] = len(fw_counts)
    # Burstiness: variance in inter-word distance
    if fw_counts:
        intervals = []
        for i, t in enumerate(tokens):
            if t in function_words:
                intervals.append(i)
        if len(intervals) > 1:
            diffs = np.diff(intervals)
            features['fw_burstiness'] = np.std(diffs) / np.mean(diffs) if np.mean(diffs) > 0 else 0
        else:
            features['fw_burstiness'] = 0
    else:
        features['fw_burstiness'] = 0

    # POS-tagged features (using nltk)
    tags = nltk.pos_tag(tokens)
    tag_counts = Counter(tag for _, tag in tags)
    n = len(tags)
    for tag in ['NN','NNS','NNP','VB','VBD','VBG','VBN','VBP','VBZ','JJ','JJR','JJS','RB','RBR','RBS','IN','DT','PRP','PRP$','CC']:
        features[f'pos_{tag}'] = tag_counts.get(tag, 0) / n if n else 0

    # First-person markers
    features['first_person'] = tokens.count('i') / n if n else 0
    features['embodied_sensation'] = sum(1 for t in tokens if t in ['felt','touch','warm','cold','smell','taste','hear','sound','skin','hand','eye','mouth'])

    # Punctuation entropy
    punct_counts = Counter(t for t in tokens if not t.isalnum())
    total_punct = sum(punct_counts.values())
    if total_punct > 0 and len(punct_counts) > 1:
        probs = np.array(list(punct_counts.values())) / total_punct
        features['punct_entropy'] = -np.sum(probs * np.log2(probs))
    else:
        features['punct_entropy'] = 0

    return features

# Extract features for human corpus
print('Extracting features from human corpus...')
human_features = []
for _, row in human_df.iterrows():
    feats = extract_features(row['text'])
    feats['id'] = row['id']
    feats['label'] = 'human'
    feats['author'] = row['author']
    feats['title'] = row['title']
    feats['year'] = row['year']
    human_features.append(feats)

human_feat_df = pd.DataFrame(human_features)
print(f'Extracted {len(human_feat_df)} feature rows, {len(human_feat_df.columns)} columns')

# Extract features for synthetic corpus
print('\nExtracting features from synthetic corpus...')
synth_features = []
for _, row in synth_df.iterrows():
    feats = extract_features(row['text'])
    feats['id'] = row['id']
    feats['label'] = 'synthetic'
    feats['prompt_id'] = row['prompt_id']
    feats['model'] = row['model']
    synth_features.append(feats)

synth_feat_df = pd.DataFrame(synth_features)
print(f'Extracted {len(synth_feat_df)} feature rows')

# Combine
all_feat_df = pd.concat([human_feat_df, synth_feat_df], ignore_index=True)
print(f'\nTotal corpus: {len(all_feat_df)} passages')
print(f'  Human: {(all_feat_df["label"] == "human").sum()}')
print(f'  Synthetic: {(all_feat_df["label"] == "synthetic").sum()}')

# Save features
feature_cols = [c for c in all_feat_df.columns if c not in ['id','label','author','title','year','prompt_id','model']]
all_feat_df.to_csv('features.csv', index=False)
print(f'\nSaved {len(feature_cols)} features to features.csv')
print(f'Feature columns: {feature_cols}')


## CELL 5/6: Stylometric analysis (Burrows' Delta + clustering)

In [ ]:
from collections import Counter
from nltk.tokenize import word_tokenize
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

def get_function_word_freqs(text, vocab):
    tokens = word_tokenize(text.lower())
    counts = Counter(tokens)
    n = len(tokens)
    return np.array([counts.get(w, 0) / n if n else 0 for w in vocab])

# Build vocabulary: top 100 most frequent function words
fw_list = ['the','be','to','of','and','a','in','that','have','i','it','for','not','on','with','he','as','you','do','at','this','but','his','by','from','they','we','say','her','she','or','an','will','my','one','all','would','there','their','what','so','up','out','if','about','who','get','which','go','me','when','make','can','like','time','no','just','him','know','take','people','into','year','your','good','some','could','them','see','other','than','then','now','look','only','come','its','over','think','also','back','after','use','two','how','our','work','first','well','way','even','new','want','because','any','these','give','day','most']
vocab = fw_list[:100]

# Get frequency vectors
freq_vectors = []
labels = []
for _, row in human_df.iterrows():
    freq_vectors.append(get_function_word_freqs(row['text'], vocab))
    labels.append(f'H:{row["author"]}')

for _, row in synth_df.iterrows():
    freq_vectors.append(get_function_word_freqs(row['text'], vocab))
    labels.append(f'S:{row["model"]}:{row["prompt_id"]}')

freq_vectors = np.array(freq_vectors)

# Z-score normalize
scaler = StandardScaler()
freq_z = scaler.fit_transform(freq_vectors)

# Burrows' Delta (Euclidean on z-scored frequencies)
delta_matrix = squareform(pdist(freq_z, metric='euclidean'))
print(f'Delta matrix shape: {delta_matrix.shape}')
print(f'Mean delta (within-human): {np.mean([delta_matrix[i,j] for i in range(len(human_df)) for j in range(len(human_df)) if i < j]):.3f}')
print(f'Mean delta (within-synthetic): {np.mean([delta_matrix[i,j] for i in range(len(human_df), len(all_feat_df)) for j in range(len(human_df), len(all_feat_df)) if i < j]):.3f}')
print(f'Mean delta (human-synthetic): {np.mean([delta_matrix[i,j] for i in range(len(human_df)) for j in range(len(human_df), len(all_feat_df))]):.3f}')

# Hierarchical clustering
linkage_matrix = linkage(freq_z, method='ward')

fig, ax = plt.subplots(1, 1, figsize=(20, 8))
dendrogram(linkage_matrix, labels=labels, leaf_rotation=90, leaf_font_size=6, ax=ax, color_threshold=8)
ax.set_title('Hierarchical clustering of human and LLM passages (Burrows Delta)', fontsize=14)
ax.set_xlabel('Passage (H = human, S = synthetic)')
ax.set_ylabel('Delta distance')
plt.tight_layout()
plt.savefig('dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

# MDS 2D visualization
mds = MDS(n_components=2, random_state=42, dissimilarity='precomputed', normalized_stress='auto')
mds_coords = mds.fit_transform(delta_matrix)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
for i, label in enumerate(labels):
    if label.startswith('H:'):
        ax.scatter(mds_coords[i, 0], mds_coords[i, 1], c='blue', alpha=0.5, s=30, label='Human' if i == 0 else '')
    else:
        ax.scatter(mds_coords[i, 0], mds_coords[i, 1], c='red', alpha=0.3, s=20, label='Synthetic' if i == len(human_df) else '')
ax.set_title('MDS projection: Human vs. Synthetic passages', fontsize=14)
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')
ax.legend()
plt.tight_layout()
plt.savefig('mds_projection.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nVisualizations saved: dendrogram.png, mds_projection.png')


## CELL 6/6: Classification + SHAP feature importance

In [ ]:
import shap

# Prepare features for classification
feature_cols = [c for c in all_feat_df.columns if c not in ['id','label','author','title','year','prompt_id','model']]
X = all_feat_df[feature_cols].fillna(0).values
y = (all_feat_df['label'] == 'synthetic').astype(int).values  # 1 = synthetic, 0 = human

print(f'Feature matrix: {X.shape}')
print(f'Class balance: {y.sum()} synthetic, {len(y) - y.sum()} human')

# Logistic Regression with stratified 5-fold CV
lr = LogisticRegression(max_iter=1000, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(lr, X, y, cv=skf, scoring='f1_macro')
print(f'\nLogistic Regression Macro-F1: {scores.mean():.3f} +/- {scores.std():.3f}')

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
scores_rf = cross_val_score(rf, X, y, cv=skf, scoring='f1_macro')
print(f'Random Forest Macro-F1: {scores_rf.mean():.3f} +/- {scores_rf.std():.3f}')

# Fit final logistic regression on all data for SHAP
lr.fit(X, y)
print(f'\nTraining LR on full data for SHAP analysis...')

# SHAP feature importance
explainer = shap.LinearExplainer(lr, X)
shap_values = explainer.shap_values(X)

feature_importance = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print(f'\nTop 15 most important features (SHAP):')
print(importance_df.head(15).to_string(index=False))

# SHAP summary plot
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
top_features = importance_df.head(15)['feature'].values
top_indices = [list(feature_cols).index(f) for f in top_features]
shap.summary_plot(
    shap_values[:, top_indices],
    X[:, top_indices],
    feature_names=top_features,
    show=False,
)
plt.title('Top 15 features distinguishing human from synthetic', fontsize=14)
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Save results
importance_df.to_csv('feature_importance.csv', index=False)
print('\nSaved feature_importance.csv')

# Save delta matrix
np.save('delta_matrix.npy', delta_matrix)
print('Saved delta_matrix.npy')

# Summary
print('\n' + '='*60)
print('ANALYSIS COMPLETE')
print('='*60)
print(f'\nGenerated artifacts:')
print(f'  - human_corpus.csv (input)')
print(f'  - synthetic_corpus.csv ({len(synth_df)} passages)')
print(f'  - features.csv (extracted features)')
print(f'  - dendrogram.png (hierarchical clustering)')
print(f'  - mds_projection.png (2D projection)')
print(f'  - shap_importance.png (feature importance)')
print(f'  - feature_importance.csv (numeric)')
print(f'  - delta_matrix.npy (pairwise distances)')
print(f'\nKey findings:')
print(f'  - Logistic Regression Macro-F1: {scores.mean():.3f}')
print(f'  - Top discriminator: {importance_df.iloc[0]["feature"]}')
print(f'  - Top 3: {", ".join(importance_df.head(3)["feature"].values)}')
